<div dir="rtl" style="text-align: right; line-height: 1.9; font-family: 'Segoe UI', Tahoma, Arial, sans-serif; font-size: 16px;">

**[🡨 بازگشت به فصل هفتم (مدل‌های پرشی و بازیابی چگالی)](Fasl_7_Masterclass.ipynb)**

# 🎓 مسترکلاس مهندسی مالی تصادفی با پایتون
## فصل ۸: قراردادهای اختیار معامله و مدل بلک-شولز (Financial Options & Black-Scholes)

---
### 🎯 هدف این نوت‌بوک
در فصول گذشته توانستیم مسیر آینده دارایی‌ها (مثل سهام) را با استفاده از **حرکت براونی هندسی (GBM)** شبیه‌سازی کنیم. در این فصل می‌خواهیم از این پیش‌بینی‌ها برای **قیمت‌گذاری مشتقات مالی (Derivatives)** به ویژه قراردادهای اختیار معامله (Options) استفاده کنیم.

**قرارداد اختیار معامله چیست؟**
* **اختیار خرید (Call Option):** حقِ (و نه اجبار) خرید یک دارایی در قیمت توافقی $K$ (Strike Price) در زمان سررسید $T$ (Expiry Time).
* **اختیار فروش (Put Option):** حقِ (و نه اجبار) فروش یک دارایی در قیمت توافقی $K$ در زمان سررسید $T$.

در این مسترکلاس می‌آموزیم:
1. **معادلات تحلیلی بلک-شولز:** پیاده‌سازی فرمول‌های بسته ریاضی برای قیمت‌گذاری دقیق.
2. **موتور شبیه‌سازی شی‌گرا:** تلفیق مدل GBM با شرایط مرزی آپشن‌ها (Payoffs).
3. **یونانی‌ها (The Greeks):** محاسبه و درک حساسیت قیمت آپشن نسبت به زمان (تتا)، نوسان (وگا)، و قیمت دارایی پایه (دلتا و گاما).
4. **مصورسازی سه‌بعدی:** رسم رویه‌های قیمتی (Option Price Surfaces).

</div>

In [ ]:
# Install necessary packages for Chapter 8
!pip install scipy numpy pandas matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm as Φ
from scipy.stats import norm
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import List, TypedDict
from enum import Enum

plt.style.use("seaborn-v0_8-darkgrid")
print("\n--- Setup Complete! Libraries for Options Pricing are loaded. ---")


--- Setup Complete! Libraries for Options Pricing are loaded. ---


<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 16px;">

### 📚 بخش ۱: ریاضیات محض بلک-شولز (Analytical Formulas)

مدل بلک-شولز با استفاده از ارزش‌گذاری خنثی نسبت به ریسک (Risk-Neutral Valuation)، قیمت منصفانه (Fair Value) یک آپشن را به صورت فرمول‌های بسته زیر ارائه می‌دهد:

مقادیر کمکی $d_1$ و $d_2$:
$$ d_1 = \frac{\ln(S_t/K) + (r + \frac{\sigma^2}{2})(T-t)}{\sigma \sqrt{T-t}} $$
$$ d_2 = d_1 - \sigma \sqrt{T-t} $$

قیمت اختیار خرید (Call Value - $V_C$) و اختیار فروش (Put Value - $V_P$):
$$ V_C(S_t, t; K, T) = S_t \Phi(d_1) - K e^{-r(T-t)} \Phi(d_2) $$
$$ V_P(S_t, t; K, T) = K e^{-r(T-t)} \Phi(-d_2) - S_t \Phi(-d_1) $$

که در آن $\Phi$ تابع توزیع تجمعی نرمال استاندارد (CDF) است.

</div>

In [ ]:
# --- 1. Black-Scholes Analytical Mathematics Engine ---

class BlackScholesOptionsGBMProcessStatistics(ABC):
    """
    Analytical (Closed-form) formulations for European Options
    and their sensitivities (Greeks).
    """
    @staticmethod
    def d1(s, t, r, σ, K, T):
        # Added small epsilon (1e-10) to prevent division by zero at t=T
        return (np.log(s/K) + (r + 0.5*(σ ** 2)) * (T-t)) / (σ * np.sqrt(T-t + 1e-10))

    @staticmethod
    def d2(s, t, r, σ, K, T):
        return BlackScholesOptionsGBMProcessStatistics.d1(s, t, r, σ, K, T) - σ * np.sqrt(T - t)

    class Call(ABC):
        @staticmethod
        def option_value(s, t, r, σ, K, T):
            d1_val = BlackScholesOptionsGBMProcessStatistics.d1(s, t, r, σ, K, T)
            d2_val = BlackScholesOptionsGBMProcessStatistics.d2(s, t, r, σ, K, T)
            return (s * Φ.cdf(d1_val)) - (K * np.exp(-r*(T-t)) * Φ.cdf(d2_val))

        @staticmethod
        def delta(s, t, r, σ, K, T):
            return Φ.cdf(BlackScholesOptionsGBMProcessStatistics.d1(s, t, r, σ, K, T))

        @staticmethod
        def gamma(s, t, r, σ, K, T):
            d1_val = BlackScholesOptionsGBMProcessStatistics.d1(s, t, r, σ, K, T)
            return Φ.pdf(d1_val) / (σ * s * np.sqrt(T-t + 1e-10))

        @staticmethod
        def theta(s, t, r, σ, K, T):
            d1_val = BlackScholesOptionsGBMProcessStatistics.d1(s, t, r, σ, K, T)
            d2_val = BlackScholesOptionsGBMProcessStatistics.d2(s, t, r, σ, K, T)
            return -((σ * s * Φ.pdf(d1_val)) / (2.0*np.sqrt(T-t + 1e-10))) - (r * K * np.exp(-r*(T-t)) * Φ.cdf(d2_val))

        @staticmethod
        def vega(s, t, r, σ, K, T):
            d1_val = BlackScholesOptionsGBMProcessStatistics.d1(s, t, r, σ, K, T)
            return Φ.pdf(d1_val) * (s * np.sqrt(T-t))

        @staticmethod
        def rho(s, t, r, σ, K, T):
            d2_val = BlackScholesOptionsGBMProcessStatistics.d2(s, t, r, σ, K, T)
            return K * (T-t) * np.exp(-r*(T-t)) * Φ.cdf(d2_val)

    class Put(ABC):
        @staticmethod
        def option_value(s, t, r, σ, K, T):
            d1_val = BlackScholesOptionsGBMProcessStatistics.d1(s, t, r, σ, K, T)
            d2_val = BlackScholesOptionsGBMProcessStatistics.d2(s, t, r, σ, K, T)
            return (K * np.exp(-r*(T-t)) * Φ.cdf(-d2_val)) - (s * Φ.cdf(-d1_val))

        @staticmethod
        def delta(s, t, r, σ, K, T):
            return Φ.cdf(BlackScholesOptionsGBMProcessStatistics.d1(s, t, r, σ, K, T)) - 1.0

        @staticmethod
        def gamma(s, t, r, σ, K, T):
            # Gamma is identical for Call and Put
            d1_val = BlackScholesOptionsGBMProcessStatistics.d1(s, t, r, σ, K, T)
            return Φ.pdf(d1_val) / (σ * s * np.sqrt(T-t + 1e-10))

print("Analytical formulas for Options and Greeks initialized.")

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 16px;">

### 📚 بخش ۲: رسم رویه آپشن‌ها (Options Surface Visualization)
قیمت یک آپشن به متغیرهای متعددی نظیر قیمت فعلی دارایی پایه ($S_t$) و زمان باقی‌مانده تا سررسید ($T-t$) وابسته است. بنابراین به جای یک خط ساده، باید خروجی را در قالب یک **سطح سه‌بعدی (3D Surface)** رسم کنیم تا هندسه قیمت‌گذاری را درک کنیم.

</div>

In [ ]:
# --- 2. 3D Surface Visualization Engine ---

def plot_options_surface(t, S, V, label, is_call=True):
    """
    Renders a stunning 3D surface plot to show how option values evolve
    with respect to underlying price S and time t.
    """
    plt.style.use("seaborn-v0_8-darkgrid")
    fig = plt.figure(figsize=(14, 6))

    # Left Plot: 3D Surface
    ax1 = fig.add_subplot(121, projection="3d")
    if is_call:
        surf = ax1.plot_surface(t, S, V, rstride=2, cstride=2, cmap=plt.cm.viridis, edgecolor="none", alpha=0.9)
        ax1.set_xlabel('Time (t)')
        ax1.set_ylabel('Stock Price (S)')
    else:
        surf = ax1.plot_surface(S, t, V, rstride=2, cstride=2, cmap=plt.cm.magma, edgecolor="none", alpha=0.9)
        ax1.set_xlabel('Stock Price (S)')
        ax1.set_ylabel('Time (t)')

    ax1.set_zlabel(label)
    ax1.set_title(f"3D Surface: {label}")
    fig.colorbar(surf, ax=ax1, shrink=0.5, aspect=10)

    # Right Plot: 2D Scatter Projection (Value vs. Price)
    ax2 = fig.add_subplot(122)
    sns.scatterplot(ax=ax2, x=S.flatten(), y=V.flatten(), hue=t.flatten(), palette="coolwarm", s=10, edgecolor=None)
    ax2.set_xlabel('Stock Price (S)')
    ax2.set_ylabel(label)
    ax2.set_title(f"2D Projection (Color = Time)")

    fig.tight_layout()
    plt.show()

def run_analytical_surface_test():
    n_points = 50
    T = 100
    ts = np.arange(T)
    S_range = np.linspace(2000, 5000, n_points)

    # Create a 2D meshgrid for time and price
    time_grid, s_grid = np.meshgrid(ts, S_range)

    # Market Parameters (e.g., S&P 500 derived)
    r, σ = 0.01, 0.04

    print("--- Call Option Payoff Surface (Strike K=3000) ---")
    V_call = [BlackScholesOptionsGBMProcessStatistics.Call.option_value(s=s, t=t, r=r, σ=σ, T=T, K=3000)
              for s, t in zip(s_grid, time_grid)]
    plot_options_surface(t=np.array(time_grid), S=np.array(s_grid), V=np.array(V_call), label="Call Value ($V_C$)", is_call=True)

    print("\n--- Put Option Payoff Surface (Strike K=3000) ---")
    V_put = [BlackScholesOptionsGBMProcessStatistics.Put.option_value(s=s, t=t, r=r, σ=σ, T=T, K=3000)
             for s, t in zip(s_grid, time_grid)]
    plot_options_surface(t=np.array(time_grid), S=np.array(s_grid), V=np.array(V_put), label="Put Value ($V_P$)", is_call=False)

run_analytical_surface_test()

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 16px;">

### 📚 بخش ۳: حساسیت‌های آپشن (یونانی‌ها - The Greeks)

تحلیلگران کمی (Quants) برای پوشش ریسک سبد سرمایه‌گذاری (Hedging) نیازمند درک دقیق حساسیت قیمت آپشن به پارامترهای مختلف هستند. به این حساسیت‌ها که مشتقات جزئی (Partial Derivatives) تابع قیمت آپشن می‌باشند، **یونانی‌ها** می‌گویند:

۱. **دلتا ($\Delta = \frac{\partial V}{\partial S}$):** نشان می‌دهد اگر دارایی پایه ۱ دلار تغییر کند، قیمت آپشن چقدر تغییر می‌کند.
۲. **گاما ($\Gamma = \frac{\partial^2 V}{\partial S^2}$):** نشان‌دهنده تقعر (Convexity) یا نرخ تغییرات دلتا است.
۳. **تتا ($\Theta = \frac{\partial V}{\partial t}$):** نشان‌دهنده استهلاک زمانی (Time Decay) ارزش آپشن با نزدیک شدن به زمان سررسید است.

در کد زیر نمودارهای سه‌بعدی این مشتقات حساسیت را ترسیم می‌کنیم.

</div>

In [ ]:
# --- 3. The Greeks Visualization ---

def run_greeks_surface_test():
    n_points = 50
    T = 100
    ts = np.arange(T)
    S_range = np.linspace(2000, 5000, n_points)
    time_grid, s_grid = np.meshgrid(ts, S_range)
    r, σ = 0.01, 0.04
    K_Strike = 3500

    print("--- 1. DELTA (Δ) for Call Option ---")
    # Delta approaches 1 when deeply In-The-Money (S >> K), and 0 when Out-Of-The-Money (S << K)
    V_delta = [BlackScholesOptionsGBMProcessStatistics.Call.delta(s=s, t=t, r=r, σ=σ, T=T, K=K_Strike)
               for s, t in zip(s_grid, time_grid)]
    plot_options_surface(t=np.array(time_grid), S=np.array(s_grid), V=np.array(V_delta), label="Delta (Δ)", is_call=True)

    print("\n--- 2. GAMMA (Γ) for Call/Put Option ---")
    # Gamma peaks At-The-Money (S = K) near expiry time.
    V_gamma = [BlackScholesOptionsGBMProcessStatistics.Call.gamma(s=s, t=t, r=r, σ=σ, T=T, K=K_Strike)
               for s, t in zip(s_grid, time_grid)]
    plot_options_surface(t=np.array(time_grid), S=np.array(s_grid), V=np.array(V_gamma), label="Gamma (Γ)", is_call=True)

    print("\n--- 3. THETA (Θ) for Call Option (Time Decay) ---")
    # Theta is usually negative because options lose value as time passes
    V_theta = [BlackScholesOptionsGBMProcessStatistics.Call.theta(s=s, t=t, r=r, σ=σ, T=T, K=K_Strike)
               for s, t in zip(s_grid, time_grid)]
    plot_options_surface(t=np.array(time_grid), S=np.array(s_grid), V=np.array(V_theta), label="Theta (Θ)", is_call=True)

run_greeks_surface_test()

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 16px;">

### 📚 بخش ۴: زیرساخت شبیه‌سازی تصادفی برای بلک-شولز (Monte Carlo Integration)

اگرچه ما فرمول‌های بسته را در بالا داریم، اما در قراردادهای پیچیده‌تر (Exotic Options) فرمول تحلیلی وجود ندارد. بنابراین ما باید از یک معماری قدرتمند شی‌گرا بر پایه **شبیه‌سازی مونت‌کارلو** استفاده کنیم.

در زیر، ما کدهای پایه‌ای فصل ۵ و ۶ (مثل `GeometricBrownianMotionProcess`) را با منطق Option در هم می‌آمیزیم تا `_BlackScholesOptionsGBMProcess` را بسازیم. این کلاس، هزاران مسیر تصادفی قیمت سهام ($S_t$) را تولید کرده و میانگین بازده‌ها را برای استخراج $V_C$ یا $V_P$ در زمان حال ($t=0$) تنزیل (Discount) می‌کند.

</div>

In [ ]:
# --- 4. Stochastic Object-Oriented Framework (Zero-Dependency) ---

# 4.1 Base Core components from previous chapters
class TargetSamplingDensity(ABC):
    @abstractmethod
    def sample(self, n_vars, n_sample_paths=1): ...

class StandardNormalTargetSamplingDensity(TargetSamplingDensity):
    def sample(self, n_vars, n_sample_paths=1): return norm.rvs(size=(n_sample_paths, n_vars))

class MonteCarloSimulation:
    @dataclass
    class MCEstimate:
        samples: List = None
        mean: np.ndarray = None
        standard_error: np.ndarray = None

    def __init__(self, h_x_fun, target_sampling_density, n_vars, n_sample_paths):
        self.h_x_fun, self.target_sampling_density = h_x_fun, target_sampling_density
        self.n_vars, self.n_sample_paths = n_vars, n_sample_paths

    def new_estimate(self):
        z = self.target_sampling_density.sample(self.n_vars, self.n_sample_paths)
        samples = np.vectorize(lambda y: self.h_x_fun(y), otypes=[float])(z)
        mean = np.average(samples, axis=0)
        variance = np.sum(np.power(samples - mean, 2), axis=0) / (self.n_sample_paths - 1 + 1e-10)
        return self.MCEstimate(samples=samples, mean=mean, standard_error=np.sqrt(variance / self.n_sample_paths))

class ForecastingProcess(ABC):
    def __init__(self, n_sample_paths, initial_state, sampling_density):
        self._n_sample_paths = n_sample_paths
        self._initial_state, self._state_t = initial_state, initial_state
        self._sampling_density = sampling_density
        self._t, self._T = 0, 0

    def forecast(self, T):
        self._T, self._t = T, 0
        mcs = MonteCarloSimulation(self._update_current_state, self._sampling_density, T, self._n_sample_paths)
        e = mcs.new_estimate()
        self._state_t = self._initial_state
        return e

    @abstractmethod
    def _update_current_state(self, z): ...

    def _reset_new_sample_path_state(self):
        if self._t >= self._T:
            self._state_t = self._initial_state
            self._t = 0
        self._t += 1

class GeometricBrownianMotionProcess(ForecastingProcess):
    def __init__(self, r, σ, initial_state=1.0, n_sample_paths=5):
        self._r, self._σ = r, σ
        self._μ = r - (σ ** 2 / 2.0)  # Ito's Drift Adjustment
        super().__init__(initial_state=initial_state, n_sample_paths=n_sample_paths,
                         sampling_density=StandardNormalTargetSamplingDensity())

    def _update_current_state(self, z):
        self._reset_new_sample_path_state()
        self._state_t = self._state_t * np.exp(self._μ + (self._σ * z))
        return self._state_t

# 4.2 Black-Scholes Options Classes
@dataclass
class OptionsResult:
    value_at_0: float = None
    all_values: object = None
    underlying_s_values: np.ndarray = None
    label: str = None
    for_call_option: bool = True

class _BlackScholesOptionsGBMProcess(GeometricBrownianMotionProcess, ABC):
    """Abstract class for simulating both the underlying asset AND the option value/greeks."""
    def __init__(self, r, σ, strike_price_K, expiry_time_T, initial_state=1.0, n_sample_paths=5):
        super().__init__(r=r, σ=σ, initial_state=initial_state, n_sample_paths=n_sample_paths)
        self._underlying_s_values = np.ndarray(shape=(n_sample_paths, expiry_time_T+1), dtype=float)
        self._current_sample_path = 0
        self._strike_price_K = strike_price_K
        self._expiry_time_T = expiry_time_T
        self._initial_v_state = self._pay_off_and_greek_f(self._initial_state, self._t)
        self._v_state = self._initial_v_state

    def _update_current_state(self, z):
        super()._update_current_state(z)  # Update Underlying S_t
        self._underlying_s_values[self._current_sample_path][self._t] = self._state_t
        self._v_state = self._pay_off_and_greek_f(self._state_t, self._t) # Calculate V_t based on S_t
        return self._v_state

    def _reset_new_sample_path_state(self):
        if self._t >= self._T:
            self._state_t = self._initial_state
            self._v_state = self._pay_off_and_greek_f(self._initial_state, 0)
            self._t = 0
            self._current_sample_path += 1
        self._t += 1

    def _d1(self, s_t, t):
        return (np.log(s_t/self._strike_price_K) + (self._r + 0.5*(self._σ ** 2)) * (self._expiry_time_T-t)) / \
               (self._σ * np.sqrt(self._expiry_time_T-t + 1e-10))
    def _d2(self, s_t, t):
        return self._d1(s_t, t) - self._σ * np.sqrt(self._expiry_time_T - t)

    @property
    def value_at_0(self): return self._initial_v_state
    @property
    def underlying_s_values(self): return self._underlying_s_values

    @abstractmethod
    def _pay_off_and_greek_f(self, s_t, t): ...
    @property
    @abstractmethod
    def label(self): ...

class _CallOptionsGBMProcess(_BlackScholesOptionsGBMProcess):
    def _pay_off_and_greek_f(self, s_t, t):
        d1, d2 = self._d1(s_t, t), self._d2(s_t, t)
        return (s_t * Φ.cdf(d1)) - (self._strike_price_K * np.exp(-self._r*(self._expiry_time_T-t)) * Φ.cdf(d2))
    @property
    def label(self): return "Call Option V(S,t)"

class BlackScholesOptionsRiskNeutralGBMModel:
    def __init__(self, parameters, n_sample_paths=5):
        self._parameters = parameters
        self._n_sample_paths = n_sample_paths

    def estimate_call(self, expiry_time_T, strike_price_K) -> OptionsResult:
        call_process = _CallOptionsGBMProcess(
            r=self._parameters['r'], σ=self._parameters['σ'],
            initial_state=self._parameters['s0'], expiry_time_T=expiry_time_T,
            strike_price_K=strike_price_K, n_sample_paths=self._n_sample_paths)

        result = call_process.forecast(T=expiry_time_T)
        return OptionsResult(all_values=result, value_at_0=call_process.value_at_0,
                             underlying_s_values=call_process.underlying_s_values,
                             label=call_process.label, for_call_option=True)

print("Object-Oriented Stochastic Engine for Black-Scholes is Ready!")

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 16px;">

### 📚 بخش ۵: اجرای شبیه‌سازی و مشاهده خروجی
اکنون مدل شی‌گرای خود را مقداردهی کرده و آن را برای شبیه‌سازی مسیرهای قیمتی یک «قرارداد اختیار خرید» (Call Option) در یک بازه ۱۰۰ روزه به کار می‌بریم.
همانطور که در نتایج خواهید دید، مدل به طور اتوماتیک `Value at 0` (همان פרمیوم یا هزینه‌ای که خریدار در ابتدا باید بپردازد) را محاسبه می‌کند.

</div>

In [ ]:
def run_stochastic_simulation():
    print("--- Executing Monte Carlo Simulation for Call Option ---")
    # Market condition parameters (e.g. S&P 500 drift and volatility)
    params = {'s0': 2000.0, 'r': 0.01, 'σ': 0.04}

    bs_model = BlackScholesOptionsRiskNeutralGBMModel(parameters=params, n_sample_paths=5)

    # Strike Price = 2100 (Out-of-the-money initially), Expiry = 100 days
    call_result = bs_model.estimate_call(expiry_time_T=100, strike_price_K=2100.0)

    print(f"[+] Fair Value of Premium Paid at t=0: ${call_result.value_at_0:.2f}")

    # Visualizing the sample paths of the Option Value V_C
    samples = pd.DataFrame(call_result.all_values.samples).transpose()
    mean_path = pd.DataFrame(call_result.all_values.mean)

    plt.figure(figsize=(10, 5))
    samples.plot(ax=plt.gca(), alpha=0.5, legend=False)
    mean_path.plot(ax=plt.gca(), color='blue', linewidth=3, label='Expected Path')
    plt.title("Simulated Paths of Call Option Value ($V_C$) over Time")
    plt.xlabel("Time Steps (Days)")
    plt.ylabel("Call Option Value ($)")
    plt.show()

run_stochastic_simulation()

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 16px;">

---
### 🏁 نتیجه‌گیری فصل ۸
در این فصل ما از تئوری‌های انتزاعی احتمالات به یک محصول ملموس در مهندسی مالی (Financial Engineering) رسیدیم:
1. دریافتیم که چگونه فرمول **Black-Scholes-Merton** با فرضیات دنیای Risk-Neutral، قیمت‌گذاری عادلانه را انجام می‌دهد.
2. یاد گرفتیم با کمک معادلات مشتقات جزئی (یونانی‌ها)، ریسک پنهان در هر معامله را شناسایی و مصورسازی کنیم (سطوح 3D دلتا، گاما، وگا).
3. با توسعه معماری نرم‌افزاری خود (از فصل ۵ و ۶)، یک موتور شبیه‌سازی مونت‌کارلو برای قیمت‌گذاری ساختیم که در آینده می‌تواند برای آپشن‌های غیرمعمول و عجیب (Exotic Options) نیز استفاده شود.

در فصل ۹، نحوه حل مستقیم معادلات دیفرانسیل با مشتقات جزئی (PDE) حاکم بر آپشن‌ها را از طریق روش‌های **تفاضل محدود (Finite Differences)** بررسی خواهیم کرد.

</div>